In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report, 
                             roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
 
print("="*70)
print("LOGISTIC REGRESSION FOR LOAN APPROVAL PREDICTION")
print("="*70)


LOGISTIC REGRESSION FOR LOAN APPROVAL PREDICTION


In [21]:
# STEP 1: LOAD AND EXPLORE DATA

print("\n[STEP 1] Loading Dataset...")
df = pd.read_csv('loan_data.csv')
 
print(f"Dataset Shape: {df.shape}")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}\n")
 
# Display first few rows
print("First 5 rows of data:")
print(df.head())
 
# Check data types
print("\n\nData Types:")
print(df.dtypes)
 
# Check missing values
print("\n\nMissing Values:")
print(df.isnull().sum())
 
# Display basic statistics
print("\n\nBasic Statistics:")
print(df.describe())
 
# Check target variable distribution
print("\n\nTarget Variable Distribution (loan_status):")
print(df['loan_status'].value_counts())
print(f"\nApproved (1): {(df['loan_status']==1).sum()} ({(df['loan_status']==1).sum()/len(df)*100:.2f}%)")
print(f"Rejected (0): {(df['loan_status']==0).sum()} ({(df['loan_status']==0).sum()/len(df)*100:.2f}%)")
 


[STEP 1] Loading Dataset...
Dataset Shape: (45000, 14)
Total Rows: 45000, Total Columns: 14

First 5 rows of data:
   person_age person_gender person_education  person_income  person_emp_exp  \
0        22.0        female           Master        71948.0               0   
1        21.0        female      High School        12282.0               0   
2        25.0        female      High School        12438.0               3   
3        23.0        female         Bachelor        79753.0               0   
4        24.0          male           Master        66135.0               1   

  person_home_ownership  loan_amnt loan_intent  loan_int_rate  \
0                  RENT    35000.0    PERSONAL          16.02   
1                   OWN     1000.0   EDUCATION          11.14   
2              MORTGAGE     5500.0     MEDICAL          12.87   
3                  RENT    35000.0     MEDICAL          15.23   
4                  RENT    35000.0     MEDICAL          14.27   

   loan_percent_in

In [ ]:
# STEP 2: DATA PREPROCESSING

print("[STEP 2] Data Preprocessing")

 
# Create a copy to avoid modifying original data
df_processed = df.copy()
 
# Handle missing values
print("\nHandling Missing Values...")
print(f"Missing values before: {df_processed.isnull().sum().sum()}")
 
# Fill missing values in numerical columns with median
numerical_cols = df_processed.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df_processed[col].isnull().sum() > 0:
        df_processed[col].fillna(df_processed[col].median(), inplace=True)
 
# Fill missing values in categorical columns with mode
categorical_cols = df_processed.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_processed[col].isnull().sum() > 0:
        df_processed[col].fillna(df_processed[col].mode()[0], inplace=True)
 
print(f"Missing values after: {df_processed.isnull().sum().sum()}")
 

# Encode Categorical Variables

print("\nEncoding Categorical Variables...")
 
# Dictionary to store label encoders (for reference)
label_encoders = {}
 
# Identify categorical columns (excluding target variable)
categorical_features = df_processed.select_dtypes(include=['object']).columns
 
for col in categorical_features:
    print(f"  - Encoding '{col}': {df_processed[col].unique().tolist()}")
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le
 
print(f"\nCategorical variables encoded: {list(categorical_features)}")

[STEP 2] Data Preprocessing

Handling Missing Values...
Missing values before: 0
Missing values after: 0

Encoding Categorical Variables...
  - Encoding 'person_gender': ['female', 'male']
  - Encoding 'person_education': ['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']
  - Encoding 'person_home_ownership': ['RENT', 'OWN', 'MORTGAGE', 'OTHER']
  - Encoding 'loan_intent': ['PERSONAL', 'EDUCATION', 'MEDICAL', 'VENTURE', 'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
  - Encoding 'previous_loan_defaults_on_file': ['No', 'Yes']

Categorical variables encoded: ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']


In [25]:
# STEP 3: SEPARATE FEATURES AND TARGET

print("[STEP 3] Preparing Features and Target")

 
# X = Features (all columns except loan_status)
# y = Target (loan_status)
X = df_processed.drop('loan_status', axis=1)
y = df_processed['loan_status']
 
print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")
 

[STEP 3] Preparing Features and Target

Features shape: (45000, 13)
Target shape: (45000,)

Feature columns: ['person_age', 'person_gender', 'person_education', 'person_income', 'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'previous_loan_defaults_on_file']


In [26]:
# STEP 4: SPLIT DATA INTO TRAINING AND TESTING SETS

print("[STEP 4] Train-Test Split")

 
# 80-20 split: 80% training, 20% testing
# random_state=42 ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"Training/Testing ratio: {len(X_train)/len(X_test):.2f}")
 
print(f"\nTraining set - Approved: {(y_train==1).sum()}, Rejected: {(y_train==0).sum()}")
print(f"Testing set - Approved: {(y_test==1).sum()}, Rejected: {(y_test==0).sum()}")

[STEP 4] Train-Test Split

Training set size: 36000 samples
Testing set size: 9000 samples
Training/Testing ratio: 4.00

Training set - Approved: 8000, Rejected: 28000
Testing set - Approved: 2000, Rejected: 7000


In [27]:
# STEP 5: FEATURE SCALING

print("[STEP 5] Feature Scaling (Standardization)")
 
# StandardScaler: Transforms features to have mean=0 and std=1
# This helps logistic regression converge faster
scaler = StandardScaler()
 
# Fit scaler on training data and transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
 
print("\nFeatures scaled using StandardScaler")
print(f"Mean of scaled training features: {X_train_scaled.mean(axis=0)[:5]} (should be ~0)")
print(f"Std of scaled training features: {X_train_scaled.std(axis=0)[:5]} (should be ~1)")
 
# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)
 

[STEP 5] Feature Scaling (Standardization)

Features scaled using StandardScaler
Mean of scaled training features: [ 2.36847579e-17 -1.47634991e-16  2.58558607e-17 -3.69087477e-17
  3.51323908e-17] (should be ~0)
Std of scaled training features: [1. 1. 1. 1. 1.] (should be ~1)


In [28]:
# STEP 6: TRAIN LOGISTIC REGRESSION MODEL

print("[STEP 6] Training Logistic Regression Model")

 
# Initialize the logistic regression model
# max_iter: Maximum iterations for optimization algorithm
# random_state: For reproducibility
# class_weight='balanced': Handles imbalanced classes
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)
 
# Train the model on training data
print("\nTraining model...")
model.fit(X_train_scaled, y_train)
print("✓ Model training completed!")
 
# Display model coefficients (feature importance)
print("\n\nModel Coefficients (Feature Importance):")
print("-" * 50)
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)
 
print(feature_importance.to_string(index=False))
print("\nNote: Positive coefficient → increases loan approval probability")
print("      Negative coefficient → decreases loan approval probability")
print(f"\nModel Intercept: {model.intercept_[0]:.4f}")
 

[STEP 6] Training Logistic Regression Model

Training model...
✓ Model training completed!


Model Coefficients (Feature Importance):
--------------------------------------------------
                       Feature  Coefficient
previous_loan_defaults_on_file    -4.898257
           loan_percent_income     1.231960
                 loan_int_rate     0.974432
                     loan_amnt    -0.557623
                  credit_score    -0.439576
         person_home_ownership     0.303132
                   loan_intent    -0.285786
                    person_age     0.134154
                person_emp_exp    -0.090440
                 person_income     0.046797
    cb_person_cred_hist_length    -0.009917
                 person_gender    -0.001025
              person_education    -0.000034

Note: Positive coefficient → increases loan approval probability
      Negative coefficient → decreases loan approval probability

Model Intercept: -4.3644


In [29]:
# STEP 7: MAKE PREDICTIONS

print("[STEP 7] Making Predictions")

 
# Predictions on test set
y_pred = model.predict(X_test_scaled)          # Class labels (0 or 1)
y_pred_proba = model.predict_proba(X_test_scaled)  # Probabilities
 
print(f"\nPredictions shape: {y_pred.shape}")
print(f"Unique predictions: {np.unique(y_pred)}")
print(f"\nFirst 10 predictions (class): {y_pred[:10]}")
print(f"First 10 predictions (probability of approval): {y_pred_proba[:10, 1]}")
 

[STEP 7] Making Predictions

Predictions shape: (9000,)
Unique predictions: [0 1]

First 10 predictions (class): [0 0 0 0 1 1 0 0 0 0]
First 10 predictions (probability of approval): [1.08571963e-01 2.65746020e-04 1.03903225e-05 1.36929627e-05
 7.50595793e-01 8.20245995e-01 1.74862466e-01 9.85719612e-06
 3.71060610e-01 4.68054094e-01]


In [30]:
# STEP 8: MODEL EVALUATION

print("[STEP 8] MODEL EVALUATION")

 
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba[:, 1])
 
print("\n" + "─" * 50)
print("PERFORMANCE METRICS")
print("─" * 50)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
 
print("\n" + "─" * 50)
print("METRIC EXPLANATIONS")
print("─" * 50)
print(f"Accuracy: {accuracy*100:.2f}% of all predictions are correct")
print(f"Precision: Of {(y_pred==1).sum()} predicted approvals, {int(precision*(y_pred==1).sum())} are actually approved")
print(f"Recall: Of {(y_test==1).sum()} actual approvals, we caught {int(recall*(y_test==1).sum())}")
print(f"F1-Score: Harmonic mean of precision and recall (balance)")
print(f"ROC-AUC: Overall model performance (higher is better, max=1.0)")
 

[STEP 8] MODEL EVALUATION

──────────────────────────────────────────────────
PERFORMANCE METRICS
──────────────────────────────────────────────────
Accuracy:  0.8487 (84.87%)
Precision: 0.6051
Recall:    0.9185
F1-Score:  0.7295
ROC-AUC:   0.9514

──────────────────────────────────────────────────
METRIC EXPLANATIONS
──────────────────────────────────────────────────
Accuracy: 84.87% of all predictions are correct
Precision: Of 3036 predicted approvals, 1837 are actually approved
Recall: Of 2000 actual approvals, we caught 1837
F1-Score: Harmonic mean of precision and recall (balance)
ROC-AUC: Overall model performance (higher is better, max=1.0)


In [32]:
# Confusion Matrix

print("CONFUSION MATRIX")
 
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
 
print(f"\n                Predicted")
print(f"                Rejected  Approved")
print(f"Actual Rejected {tn:6d}    {fp:6d}")
print(f"       Approved {fn:6d}    {tp:6d}")
 
print(f"\nTrue Negatives (TN):  {tn} - Correctly predicted rejections")
print(f"False Positives (FP): {fp} - Incorrectly approved loans (risky!)")
print(f"False Negatives (FN): {fn} - Incorrectly rejected loans (lost opportunities)")
print(f"True Positives (TP):  {tp} - Correctly predicted approvals")
 
# Classification Report
print("\n\n" + "─" * 50)
print("CLASSIFICATION REPORT")
print("─" * 50)
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))
 


CONFUSION MATRIX

                Predicted
                Rejected  Approved
Actual Rejected   5801      1199
       Approved    163      1837

True Negatives (TN):  5801 - Correctly predicted rejections
False Positives (FP): 1199 - Incorrectly approved loans (risky!)
False Negatives (FN): 163 - Incorrectly rejected loans (lost opportunities)
True Positives (TP):  1837 - Correctly predicted approvals


──────────────────────────────────────────────────
CLASSIFICATION REPORT
──────────────────────────────────────────────────
              precision    recall  f1-score   support

    Rejected       0.97      0.83      0.89      7000
    Approved       0.61      0.92      0.73      2000

    accuracy                           0.85      9000
   macro avg       0.79      0.87      0.81      9000
weighted avg       0.89      0.85      0.86      9000



In [33]:
# STEP 9: VISUALIZATIONS

print("[STEP 9] Creating Visualizations")
 
# Figure 1: Confusion Matrix Heatmap
plt.figure(figsize=(10, 8))
 
# 1. Confusion Matrix
plt.subplot(2, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Rejected', 'Approved'],
            yticklabels=['Rejected', 'Approved'])
plt.title('Confusion Matrix', fontsize=12, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
 
# 2. ROC Curve
plt.subplot(2, 2, 2)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba[:, 1])
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve', fontsize=12, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
 
# 3. Feature Importance (Top 10)
plt.subplot(2, 2, 3)
top_features = feature_importance.head(10)
colors = ['green' if x > 0 else 'red' for x in top_features['Coefficient']]
plt.barh(range(len(top_features)), top_features['Coefficient'], color=colors, alpha=0.7)
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Coefficient Value')
plt.title('Top 10 Most Important Features', fontsize=12, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(True, alpha=0.3, axis='x')
 
# 4. Prediction Distribution
plt.subplot(2, 2, 4)
plt.hist(y_pred_proba[y_test==0, 1], bins=50, alpha=0.6, label='Rejected Loans', color='red')
plt.hist(y_pred_proba[y_test==1, 1], bins=50, alpha=0.6, label='Approved Loans', color='green')
plt.xlabel('Predicted Probability of Approval')
plt.ylabel('Frequency')
plt.title('Distribution of Predicted Probabilities', fontsize=12, fontweight='bold')
plt.legend()
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=1, label='Decision Boundary')
plt.grid(True, alpha=0.3, axis='y')
 
plt.tight_layout()
plt.savefig('logistic_regression_results.png', dpi=300, bbox_inches='tight')
plt.show()


[STEP 9] Creating Visualizations


In [34]:
# STEP 10: SAMPLE PREDICTIONS

print("[STEP 10] Sample Predictions on Test Data")
 
# Display predictions for first 10 test samples
sample_df = pd.DataFrame({
    'Actual': y_test.iloc[:10].values,
    'Predicted': y_pred[:10],
    'Approval_Probability': y_pred_proba[:10, 1],
    'Correct': y_test.iloc[:10].values == y_pred[:10]
})
 
print("\nFirst 10 Test Samples:")
print(sample_df.to_string(index=False))
 


[STEP 10] Sample Predictions on Test Data

First 10 Test Samples:
 Actual  Predicted  Approval_Probability  Correct
      0          0              0.108572     True
      0          0              0.000266     True
      0          0              0.000010     True
      0          0              0.000014     True
      1          1              0.750596     True
      1          1              0.820246     True
      0          0              0.174862     True
      0          0              0.000010     True
      0          0              0.371061     True
      0          0              0.468054     True


In [ ]:
# SUMMARY AND RECOMMENDATIONS

print("SUMMARY AND CONCLUSIONS")

 
print(f"""
✓ Model Successfully Built and Trained!
 
KEY FINDINGS:
─────────────
• Model Accuracy: {accuracy*100:.2f}%
• ROC-AUC Score: {roc_auc:.4f}
• Precision: {precision:.4f} (Reliability of approved loans)
• Recall: {recall:.4f} (Coverage of approvals)
 
FEATURE IMPORTANCE (Top 5):
─────────────────────────
""")
for idx, row in feature_importance.head(5).iterrows():
    print(f"  {row['Feature']:30s}: {row['Coefficient']:7.4f}")
 
print(f"""
BUSINESS INSIGHTS:
──────────────────
1. False Positives (FP={fp}): Approved loans that defaulted → Revenue Risk
2. False Negatives (FN={fn}): Rejected loans that would have been good → Opportunity Cost
3. Decision Threshold: Currently 0.5 (can be adjusted based on risk tolerance)
 
NEXT STEPS FOR IMPROVEMENT:
───────────────────────────
1. Analyze why certain loans were misclassified
2. Engineer new features (income-to-loan ratio, etc.)
3. Try different probability thresholds for business optimization
4. Compare with other models (Random Forest, Gradient Boosting)
5. Implement cross-validation for more robust evaluation
6. Handle class imbalance if present in data
""")
 
print("MODEL TRAINING COMPLETE!")

 

SUMMARY AND CONCLUSIONS

✓ Model Successfully Built and Trained!

KEY FINDINGS:
─────────────
• Model Accuracy: 84.87%
• ROC-AUC Score: 0.9514
• Precision: 0.6051 (Reliability of approved loans)
• Recall: 0.9185 (Coverage of approvals)

FEATURE IMPORTANCE (Top 5):
─────────────────────────

  previous_loan_defaults_on_file: -4.8983
  loan_percent_income           :  1.2320
  loan_int_rate                 :  0.9744
  loan_amnt                     : -0.5576
  credit_score                  : -0.4396

BUSINESS INSIGHTS:
──────────────────
1. False Positives (FP=1199): Approved loans that defaulted → Revenue Risk
2. False Negatives (FN=163): Rejected loans that would have been good → Opportunity Cost
3. Decision Threshold: Currently 0.5 (can be adjusted based on risk tolerance)

NEXT STEPS FOR IMPROVEMENT:
───────────────────────────
1. Analyze why certain loans were misclassified
2. Engineer new features (income-to-loan ratio, etc.)
3. Try different probability thresholds for business opti